# 01 — LlamaIndex Basics

**Goal**: Understand LlamaIndex's core concepts: Documents, Nodes, and Indices.

LlamaIndex specializes in **connecting data to LLMs**. It's the go-to framework for RAG.

In [ ]:
# !pip install llama-index llama-index-llms-ollama llama-index-embeddings-ollama

from llama_index.core import Document, VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

print("Imports OK")

## 1. LlamaIndex Core Concepts

```
Documents (your files)
     │
     ▼
   [Parser]  ← splits into chunks
     │
     ▼
   Nodes (chunks with metadata)
     │
     ▼
   [Embedding Model]
     │
     ▼
   Index (vector store)
     │
     ▼
   Query Engine (retrieve + synthesize)
```

## 2. Creating Documents

A Document = any piece of text with metadata.

In [ ]:
docs = [
    Document(text="The transformer architecture introduced self-attention, allowing models to process all tokens in parallel. This was a breakthrough over RNNs.", metadata={"source": "paper", "year": 2017}),
    Document(text="RAG combines a retrieval system with an LLM. The retriever finds relevant documents, and the LLM generates answers based on them.", metadata={"source": "tutorial", "topic": "rag"}),
    Document(text="Fine-tuning adapts a pretrained model by continuing training on domain-specific data. LoRA makes this efficient by adding small adapter weights.", metadata={"source": "guide", "topic": "fine-tuning"}),
    Document(text="Vector databases store embeddings and enable efficient similarity search. They're essential for production RAG systems.", metadata={"source": "blog", "topic": "infrastructure"}),
]

print(f"Created {len(docs)} documents")

## 3. Parsing Documents into Nodes

Nodes are chunks of text with rich metadata. Chunking strategy matters!

In [ ]:
parser = SimpleNodeParser.from_defaults(chunk_size=512, chunk_overlap=20)
nodes = parser.get_nodes_from_documents(docs)

print(f"Documents: {len(docs)} → Nodes: {len(nodes)}")
print("\nFirst node:")
print(f"  Text: {nodes[0].text[:80]}...")
print(f"  Metadata: {nodes[0].metadata}")

## 4. Building a Vector Index

This automatically: embeds all nodes → stores them in an in-memory vector store → creates a query engine.

In [ ]:
# Set up Ollama LLM and embedding
llm = Ollama(model="llama3.2", request_timeout=60.0)
embed_model = OllamaEmbedding(model_name="nomic-embed-text")

# Build index
index = VectorStoreIndex.from_documents(
    docs,
    llm=llm,
    embed_model=embed_model
)

print("Index built!")

## 5. Querying

The query engine: retrieves relevant nodes → synthesizes answer using LLM.

In [ ]:
query_engine = index.as_query_engine()

response = query_engine.query("What is the transformer architecture and why was it important?")
print(f"Answer: {response}")
print(f"\nSource nodes: {len(response.source_nodes)}")
for sn in response.source_nodes:
    print(f"  - {sn.node.text[:60]}... (score: {sn.score:.3f})")

In [ ]:
response = query_engine.query("How does LoRA make fine-tuning efficient?")
print(response)

## 6. Chunk Size Effects

Different chunk sizes give different results.

In [ ]:
def test_chunk_size(size):
    parser = SimpleNodeParser.from_defaults(chunk_size=size, chunk_overlap=20)
    nodes = parser.get_nodes_from_documents(docs)
    return len(nodes)

for size in [128, 256, 512, 1024]:
    n = test_chunk_size(size)
    print(f"Chunk size {size}: {n} nodes")

## Key Takeaways

1. **Documents** → **Nodes** → **Index** → **Query Engine** (the LlamaIndex pipeline)
2. **Chunking** matters — smaller chunks = more precise, larger = more context
3. **LlamaIndex** handles embedding + retrieval + synthesis automatically
4. Source nodes show you *why* the LLM gave that answer (traceability)

**Next**: Build a full RAG pipeline with your own data →